<a href="https://colab.research.google.com/github/SFcrypt/ColabUI/blob/main/Maker/SwarmUI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 🤖 **Instalar entorno**

</details><img src='https://i.pinimg.com/originals/d4/63/f2/d463f24b0e1f3f1ce6680d601c97e6a0.gif' width='700px'/>

<details>
<summary><font color=gray>Últimos cambios</summary>

```diff
> ☢️ Importante: ejecute la opción indicada antes que nada.
> ☢️ Importante: tenga disponible la información necesaria.
> ☢️ Importante: no es necesario reiniciar el sistema.
> 🟢 SageMaker: mantendrá la instalación completada.
```
</details>

In [ ]:

Webui = 'SwarmUI' # @param ["A1111", "Forge", "ReForge", "ReForge-old", "Forge-Classic", "Forge-Neo", "ComfyUI", "SwarmUI"]
Civitai___Key = '46c6a8c6f201dcd90900d715fdbaa728' # @param { type: "string", placeholder: "Your Civitai API Key (required)" }
HF_Read_Token = ''
Mount__GDrive = 'No' # @param ["Yes", "No"]

mount = Mount__GDrive

if mount == 'Yes':
    from google.colab import drive
    drive.mount('/content/drive')

!curl -sLo /content/setup.py https://github.com/gutris1/segsmaker/raw/main/script/KC/setup.py
%run /content/setup.py --webui="$Webui" --civitai_key="$Civitai___Key" --hf_read_token="$HF_Read_Token"

if mount == 'Yes':
    from pathlib import Path

    d = Path('/content/drive/MyDrive/Segsmaker')

    for n, p in {'checkpoint': CKPT, 'lora': LORA, 'vae': VAE, 'embeddings': Embeddings}.items():
        f = d / n
        f.mkdir(parents=True, exist_ok=True)
        s = p / f'drive-{n}'
        s.symlink_to(f, target_is_directory=True)

    !rm -rf $WebUI_Output
    o = d / {'ComfyUI': 'comfyui-output', 'SwarmUI': 'swarmui-output'}.get(Webui, 'output')
    o.mkdir(parents=True, exist_ok=True)
    WebUI_Output.symlink_to(o, target_is_directory=True)

    if Webui not in {'ComfyUI', 'SwarmUI'}:
        wc = WebUI / 'cache'
        !rm -rf $wc
        c = d / 'cache'
        c.mkdir(parents=True, exist_ok=True)
        wc.symlink_to(c, target_is_directory=True)

In [ ]:
#@markdown #🤖 <b><font color='grey'>ComfyUI

import os
os.makedirs("/content/SwarmUI/dlbackend", exist_ok=True)

# Eliminar archivo específico si existe
file_to_delete = "/content/SwarmUI/Models/Embeddings/unaestheticXL_cbp62 -neg.safetensors"
if os.path.isfile(file_to_delete):
    os.remove(file_to_delete)

# Clonar ComfyUI
%cd -q /content/SwarmUI/dlbackend
!git clone https://github.com/SFcrypt/ComfyUI

# Actualizar SwarmUI sobrescribiendo archivos
%cd -q /content/
!git clone https://github.com/SFcrypt/SwarmUI SwarmUI_tmp
!cp -r -f SwarmUI_tmp/* SwarmUI/
!rm -rf SwarmUI_tmp

#Fin

In [ ]:
#@markdown # 🗿 <b><font color='grey'>Controlnet

%cd -q /content/SwarmUI/Models/controlnet
%download https://huggingface.co/SFcrypt/controlnet-union/resolve/main/union.safetensors

## 🦜 **Instalar modelo**

</details><img src='https://i.pinimg.com/originals/d4/63/f2/d463f24b0e1f3f1ce6680d601c97e6a0.gif' width='700px'/>

<details>
<summary><font color=gray>Últimos cambios</summary>
```diff
+ Descarga Modelo Pony Versión 6
+ Descarga Imágenes Pony Versión 6
```
</details>

In [ ]:
#@markdown # 🐊 <b><font color='grey'>Checkpoint
Modelo = "Nova Reality XL" #@param ["Anime", "Blender", "Nova Reality XL"]

%cd -q $CKPT

if Modelo == "Nova Reality XL":
    %download https://huggingface.co/SFcrypt/illustrious-s/resolve/main/Nova.png Nova-reality.preview.png
    %download https://huggingface.co/SFcrypt/illustrious-s/resolve/main/Nova.saf Nova-reality.safetensors
elif Modelo == "Anime":
    %download https://civitai.com/api/download/models/2167369 WAI-illustrious.safetensors
elif Modelo == "Blender":
    %download https://civitai.com/api/download/models/1637657 BlenderXL.safetensors

In [ ]:
#@markdown # 🐌 <b><font color='grey'>LoRa huggingface

%cd -q $LORA

!git clone https://huggingface.co/SFcrypt/illustrious-t
!cp -r illustrious-t/* .
!rm -rf illustrious-t

In [ ]:
#@markdown # 🦜 <b><font color='grey'>LoRa Personalizado
import os, urllib.request, ipywidgets as widgets
from IPython.display import display, clear_output

# clonar repositorio si no existe
%cd -q $HOME
if not os.path.exists("ColabUI"):
    !git clone https://github.com/SFcrypt/ColabUI.git
    clear_output()

# Cargar CSS segsmaker
from ColabUI.Widgets.segsmaker_box import load_segsmaker_style, segsmaker_box
load_segsmaker_style()

# Inputs con margen y placeholder opaco
link_input = widgets.Text(
    placeholder="Link de descarga",
    layout=widgets.Layout(margin="5px 0 10px 0"))

link_input.add_class("seg-input")
link_input.style.placeholder_color = '#d0d0d099'

nombre_input = widgets.Text(
    placeholder="Nombre del LoRa",  # indicamos que será minúsculas
    layout=widgets.Layout(margin="5px 0 15px 0"))
nombre_input.add_class("seg-input")
nombre_input.style.placeholder_color = '#d0d0d099'

download_btn = widgets.Button(description="Download")
download_btn.add_class("seg-button")

# Función de descarga
def descargar_lora(b):
    clear_output(wait=True)
    Link, Nombre = link_input.value.strip(), nombre_input.value.strip()
    if not Link:
        return  # nada que descargar

    # Convertir el nombre a minúsculas y formatear
    filename = f"{'-'.join(Nombre.lower().split())}.safetensors" if Nombre else ""

    # Cambiar directorio a $LORA y descargar
    %cd -q $LORA
    get_ipython().run_line_magic("download", f"{Link} {filename}".strip())

download_btn.on_click(descargar_lora)

# Mostrar widget
segsmaker_box(
    content=[link_input, nombre_input, download_btn],
    width="360px")

#Fin

## 🎮 **Iniciar entorno**

</details><img src='https://i.pinimg.com/originals/d4/63/f2/d463f24b0e1f3f1ce6680d601c97e6a0.gif' width='700px'/>

<details>
<summary><font color=gray>Últimos cambios</summary>
-  **A1111** = `--xformers`
- **Forge** = `--disable-xformers --opt-sdp-attention --cuda-stream`
- **ReForge** = `--xformers --cuda-stream`
- **Forge-Classic** = `--xformers --cuda-stream --persistent-patches`
- **Forge-Neo** = `--xformers --cuda-stream`
- **ComfyUI** = `--dont-print-server --use-pytorch-cross-attention`
- **SwarmUI** = `--launch_mode none`

In [ ]:
#@markdown # 🎮 <b><font color="#FFB3B3">I</font><font color="#FFD1A6">n</font><font color="#FFF2A6">i</font><font color="#C6F6C6">c</font><font color="#C6D8FF">i</font><font color="#D8C6FF">a</font><font color="#F2C6FF">r</font></b> {"display-mode":"form"}
#@markdown Prepara y ejecuta el entorno **SwarmUl**, configurando los módulos necesarios y dejando todo listo para trabajar.

%cd -q $WebUI
%run segsmaker.py --launch_mode none

## 📐 **Ajustes**

</details><img src='https://i.pinimg.com/originals/d4/63/f2/d463f24b0e1f3f1ce6680d601c97e6a0.gif' width='700px'/>

<details>
<summary><font color=gray>Últimos cambios</summary>
```diff
+ Crea un link para ejecutar
+ Ejecuta Stable diffusion
```
</details>

In [ ]:
#@markdown # 🧽 <b><font color='grey'>Limpiar LoRAs

import os
%cd -q $LORA

# Eliminar archivos que no sean .safetensors
for file in os.listdir("."):
    if os.path.isfile(file) and not file.endswith(".safetensors"):
        os.remove(file)

In [ ]:
#@markdown # 🐦 <b><font color='grey'> Modelos Personales

''' SD Extensions / ComfyUI '''
%cd -q $Extensions
!git clone

''' VAE '''
%cd -q $VAE
%download

''' Embeddings '''
%cd -q $Embeddings
%download

''' Upscalers '''
%cd -q $Upscalers
%download

''' FLUX Unet '''
%cd -q $UNET
%download

''' FLUX Clip '''
%cd -q $CLIP
%download

In [ ]:
#@markdown # 🦭 <b><font color='grey'>Controlnet

''' Controlnet '''
%run $Controlnet_Widget